# SISO CSI Capture & Plot (ZMQ)
This notebook is converted from the original Python script.

In [ ]:

# Imports
import zmq
import time
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime


## Utility Functions

In [ ]:

def close_zmq(context, publisher_socket, subscriber_socket):
    print("Closing ZMQ sockets and terminating context...")
    try:
        publisher_socket.close(0)
    except Exception:
        pass
    try:
        subscriber_socket.close(0)
    except Exception:
        pass
    try:
        context.term()
    except Exception:
        pass
    print("Done.")


def receive_data(socket):
    return socket.recv()


def deserialize_data_siso(data, dtype=np.complex64):
    return np.frombuffer(data, dtype=dtype)


def filter_vector(vec):
    return vec[(vec.real != 1) | (vec.imag != 0)]


## Save and Plot Functions

In [ ]:

def save_vector_to_file(vec, event, out_root="."):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_dir = os.path.join(out_root, f"Channel_Capture_{timestamp}_Event{event}")
    os.makedirs(base_dir, exist_ok=True)

    file_name = os.path.join(base_dir, f"siso_vector_{timestamp}.csv")
    np.savetxt(
        file_name,
        np.column_stack((vec.real, vec.imag)),
        delimiter=",",
        header="Real,Imaginary",
        comments=""
    )
    print(f"[Event {event}] saved -> {file_name}")
    return file_name


def plot_vector(vec, title="SISO Channel"):
    idx = np.arange(len(vec))
    plt.figure(figsize=(12, 6))

    plt.subplot(2, 1, 1)
    plt.plot(idx, 20 * np.log10(np.abs(vec) + 1e-12))
    plt.title("Magnitude (dB)")
    plt.xlabel("Index")
    plt.ylabel("Magnitude (dB)")

    plt.subplot(2, 1, 2)
    plt.plot(idx, np.unwrap(np.angle(vec)))
    plt.title("Phase (rad)")
    plt.xlabel("Index")
    plt.ylabel("Phase (rad)")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


## Main ZMQ Receiver Loop

In [ ]:

# Configuration
SLEEP_SECONDS = 10
PLOT_EACH_EVENT = False

context = zmq.Context()

subscriber_socket = context.socket(zmq.SUB)
subscriber_socket.connect("tcp://localhost:5555")
subscriber_socket.setsockopt_string(zmq.SUBSCRIBE, "")

publisher_socket = context.socket(zmq.PUB)
publisher_socket.bind("tcp://*:5556")  # compatibility

print("Waiting for data (SISO, 1 RX port)... Ctrl+C to stop.")
event = 0

try:
    while True:
        event += 1
        data = receive_data(subscriber_socket)

        vec = deserialize_data_siso(data, np.complex64)
        filtered = filter_vector(vec)

        print(f"[Event {event}] received {len(vec)} samples, filtered -> {len(filtered)}")

        save_vector_to_file(filtered, event)

        if PLOT_EACH_EVENT:
            plot_vector(filtered, title=f"SISO Channel - Event {event}")

        time.sleep(SLEEP_SECONDS)

except KeyboardInterrupt:
    print("Program terminated.")
finally:
    close_zmq(context, publisher_socket, subscriber_socket)
